# **SMS Spam Filter**

In [13]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [14]:
# Read dataset
spam = pd.read_csv('../data/spam.csv', encoding='latin-1')[['v1','v2']]

In [15]:
spam.columns = ['label', 'text']

In [16]:
spam.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
# Add Plotly for visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Label distribution
fig = px.histogram(spam, x='label', title='Distribution of Spam vs Ham Messages', color='label')
fig.show()

# Text length distribution
spam['text_length'] = spam['text'].apply(len)
fig = px.histogram(spam, x='text_length', color='label', title='Text Length Distribution by Label', marginal='box')
fig.show()

## **Prepare Data**

In [18]:
X = spam['text']
y = spam['label']

vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

## **Train Model**

In [19]:
model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## **Evaluate**

In [20]:
y_pred = model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       1.00      0.77      0.87       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.93      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[965   0]
 [ 35 115]]


In [21]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
fig = go.Figure(data=go.Heatmap(z=cm, x=['Predicted Ham', 'Predicted Spam'], y=['Actual Ham', 'Actual Spam'], colorscale='Blues'))
fig.update_layout(title='Confusion Matrix', xaxis_title='Predicted', yaxis_title='Actual')
fig.show()

# Prediction Probabilities
y_proba = model.predict_proba(X_test)[:, 1]
fig = px.histogram(x=y_proba, color=y_test, title='Prediction Probabilities Distribution', labels={'x': 'Predicted Probability of Spam'})
fig.show()

## **Save Model and Vectorizer**

In [22]:
import os
dump_dir = "../models/"
os.makedirs(dump_dir, exist_ok=True)

joblib.dump(model, os.path.join(dump_dir, "spam_classifier_model.joblib"))
joblib.dump(vectorizer, os.path.join(dump_dir, "tfidf_vectorizer.joblib"))
print("\nSaved Successfully.")


Saved Successfully.


## **Testing**

In [ ]:
test_texts = [
    "Congratulations! You have been selected to win a $1000 gift card!",
    "Hey, are we still on for the meeting tomorrow?",
    "Click this link to claim your exclusive reward!!!",
    "Can you send me the documents?",
    "Free entry in a weekly competition to win FA Cup final tickets!",
    "Don't forget to bring your laptop to the workshop.",
    "Limited time offer! Get 50% off on all products.",
    "Let's catch up over lunch this weekend.",
    "You have won a lottery of $10,000,000! Contact us immediately.",
]

test_vectors = vectorizer.transform(test_texts)
predictions = model.predict(test_vectors)

print("\nModel Predictions:")
for text, pred in zip(test_texts, predictions):
    print(f"{pred.upper()} --> {text}")


Model Predictions:
SPAM --> Congratulations! You have been selected to win a $1000 gift card!
HAM --> Hey, are we still on for the meeting tomorrow?
SPAM --> Click this link to claim your exclusive reward!!!
HAM --> Can you send me the documents?
SPAM --> Free entry in a weekly competition to win FA Cup final tickets!
HAM --> Don't forget to bring your laptop to the workshop.
HAM --> Limited time offer! Get 50% off on all products.
HAM --> Let's catch up over lunch this weekend.
SPAM --> You have won a lottery of $10,000,000! Contact us immediately.
